[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AllInVaders/aistudio-full-course/blob/main/notebooks/01_Setup_Models_and_Token_Economics.ipynb)

# Module 01: Google AI Studio Setup, Model Family Discovery & Token Economics
### Módulo 01: Configuración de Google AI Studio, Familias de Modelos y Economía de Tokens

**English Overview**: In this notebook, you will configure the official unified `google-genai` SDK, verify your `GEMINI_API_KEY` (or Vertex AI ADC), enumerate available **Gemini 2.5**, **Imagen 3**, and **Veo** models, and benchmark preflight token accounting (`client.models.count_tokens`).

**Resumen en Español**: En este cuaderno interactivo configurarás el SDK oficial unificado `google-genai`, verificarás tu `GEMINI_API_KEY` (o credenciales ADC de Vertex AI), explorarás los modelos disponibles de **Gemini 2.5**, **Imagen 3** y **Veo**, y medirás el consumo de tokens con `client.models.count_tokens`.

In [ ]:
# Step 1: Install the official unified Google GenAI SDK
%pip install -q -U google-genai pydantic

In [ ]:
# Step 2: Configure your API Key (works seamlessly in Google Colab or local Jupyter)
import os
from google import genai
from google.genai import types

try:
    from google.colab import userdata  # type: ignore
    os.environ.setdefault('GEMINI_API_KEY', userdata.get('GEMINI_API_KEY'))
except Exception:
    pass

client = genai.Client()
print('Initialized google-genai client successfully!')

## 1. Discovering Gemini, Imagen 3 & Veo Model Families / Descubrimiento de Modelos
We query `client.models.list()` to inspect every model accessible to your API key across text, multimodal, image generation, video generation, and embeddings.

In [ ]:
families = {'Gemini': [], 'Imagen': [], 'Veo': [], 'Embedding': []}
for m in client.models.list():
    name = (m.name or '').replace('models/', '')
    if 'imagen' in name:
        families['Imagen'].append(name)
    elif 'veo' in name:
        families['Veo'].append(name)
    elif 'embedding' in name:
        families['Embedding'].append(name)
    elif 'gemini' in name:
        families['Gemini'].append(name)

for family, items in families.items():
    print(f'{family:<10}: {len(items):>2} models -> {items[:4]}')

## 2. Preflight Token Counting & Cost Budgeting / Conteo de Tokens y Presupuesto
Before sending large multimodal prompts in production, call `client.models.count_tokens()` to verify input token counts and enforce per-request budget ceilings.

In [ ]:
prompt = (
    'You are an AI Product Studio strategist. Summarize the 3 pillars of a '
    'high-converting product launch brief in English and Spanish.'
)
token_info = client.models.count_tokens(model='gemini-2.5-flash', contents=prompt)
print('Preflight token count:', token_info.total_tokens)

response = client.models.generate_content(
    model='gemini-2.5-flash',
    contents=prompt,
    config=types.GenerateContentConfig(temperature=0.2, max_output_tokens=300),
)
print('\nUsage Metadata:', response.usage_metadata)
print('\nModel Response:\n', response.text)